# Write data to csv file

In [1]:
import xarray as xr
import pandas as pd
import numpy as np

In [2]:
%cd /g/data/w42/dr6273/work/seasonal_energy/

import functions as fn

/g/data/w42/dr6273/work/seasonal_energy


# Load data

#### Climate drivers

- 3 monthly mean for oceanic indices

In [3]:
def roll_3month(ds):
    """
    3 monthly mean
    """
    return ds.rolling(time=3).mean()

In [4]:
drivers_fp = "/g/data/w42/dr6273/work/data/"

In [5]:
nino34 = xr.open_zarr(
    drivers_fp + "hadisst/climate_modes/hadisst_nino34_1940-2023.zarr",
    consolidated=True
)

In [6]:
nino34 = roll_3month(nino34).compute()

In [7]:
dmi = xr.open_zarr(
    drivers_fp + "hadisst/climate_modes/hadisst_dmi_1940-2023.zarr",
    consolidated=True
)

In [8]:
dmi = roll_3month(dmi).compute()

In [9]:
sam = xr.open_zarr(
    drivers_fp + "era5/climate_modes/era5_sam_1940-2023.zarr",
    consolidated=True
).compute()

Can't floor timestamps to the beginning of the month, so hack it and set Nino34 and DMI time to that of SAM

In [10]:
nino34["time"] = sam["time"]

In [11]:
dmi["time"] = sam["time"]

#### Generation and demand

In [12]:
demand_current = fn.load_monthly("demand_state_training2010-16_extrapolate1940-2023")

In [13]:
demand_SC30 = fn.load_monthly("demand_state_step_change_2029-30_training2010-16_extrapolate1940-2023")

In [14]:
demand_SC40 = fn.load_monthly("demand_state_step_change_2039-40_training2010-16_extrapolate1940-2023")

In [15]:
demand_SC50 = fn.load_monthly("demand_state_step_change_2049-50_training2010-16_extrapolate1940-2023")

In [16]:
wind_current = fn.load_monthly("wind_power_REZ_current_state_van_der_Wiel_era5_monthly_1940-2023")

In [17]:
wind_SC30 = fn.load_monthly("wind_power_REZ_step_change_2029-30_state_van_der_Wiel_era5_monthly_1940-2023")

In [18]:
wind_SC40 = fn.load_monthly("wind_power_REZ_step_change_2039-40_state_van_der_Wiel_era5_monthly_1940-2023")

In [19]:
wind_SC50 = fn.load_monthly("wind_power_REZ_step_change_2049-50_state_van_der_Wiel_era5_monthly_1940-2023")

In [20]:
solar_current = fn.load_monthly("solar_power_REZ_current_state_van_der_Wiel_era5_monthly_1940-2023")

In [21]:
solar_SC30 = fn.load_monthly("solar_power_REZ_step_change_2029-30_state_van_der_Wiel_era5_monthly_1940-2023")

In [22]:
solar_SC40 = fn.load_monthly("solar_power_REZ_step_change_2039-40_state_van_der_Wiel_era5_monthly_1940-2023")

In [23]:
solar_SC50 = fn.load_monthly("solar_power_REZ_step_change_2049-50_state_van_der_Wiel_era5_monthly_1940-2023")

In [24]:
shortfall_current = fn.load_monthly("shortfall_REZ_current_state_van_der_Wiel_training2010-16_extrapolate1940-2023")

In [25]:
shortfall_SC30 = fn.load_monthly("shortfall_REZ_step_change_2029-30_state_van_der_Wiel_training2010-16_extrapolate1940-2023")

In [26]:
shortfall_SC40 = fn.load_monthly("shortfall_REZ_step_change_2039-40_state_van_der_Wiel_training2010-16_extrapolate1940-2023")

In [27]:
shortfall_SC50 = fn.load_monthly("shortfall_REZ_step_change_2049-50_state_van_der_Wiel_training2010-16_extrapolate1940-2023")

#### Other predictors

Temperature

In [28]:
t2m = xr.open_dataset(
    "/g/data/w42/dr6273/work/projects/Aus_energy/demand_predictors/2t_era5_daily_1940-2023_NEM_pop_dens_mask.nc"
)["t2m"]

In [29]:
t2m = t2m.resample(time="1MS").mean()

In [30]:
t2m = t2m.to_dataset(name="t2m")

In [31]:
t2m["t2m_detrended"] = fn.detrend_dim(t2m["t2m"], "time")

In [82]:
def to_df(detrended, region, target_list, predictor_list, scenario, anomalise_monthly=False):
    """
    Write to csv
    
    detrended: bool, whether to use detrended data or not
    region: str, region to process
    target_list: list of target datasets [demand, wind, solar, shortfall]
    predictor_list: list of predictor datasets [nino34, dmi, sam, t2m]
    scenario: str, e.g. current or SC30
    anomalise_monthly: bool, whether or not to anomalise data as a function of calendar month
    """
    if detrended:
        var_append = "_detrended"
    else:
        var_append = ""
        
    if anomalise_monthly:
        target_list = [ds.groupby("time.month").apply(lambda x: x - x.mean("time")) for ds in target_list]
        predictor_list = [ds.groupby("time.month").apply(lambda x: x - x.mean("time")) for ds in predictor_list]
        # nino34 = nino34.groupby("time.month").apply(lambda x: x - x.mean("time"))
        # dmi = dmi.groupby("time.month").apply(lambda x: x - x.mean("time"))
        # sam = sam.groupby("time.month").apply(lambda x: x - x.mean("time"))
        # t2m = t2m.groupby("time.month").apply(lambda x: x - x.mean("time"))
        
    data = [
            predictor_list[0].sel(time=slice("1941-03", None)).time.dt.year.values,
            predictor_list[0].sel(time=slice("1941-03", None)).time.dt.month.values,
            target_list[0]["demand" + var_append].sel(time=slice("1941-03", None)).sel(region=region).values,
            target_list[1]["power" + var_append].sel(time=slice("1941-03", None)).sel(region=region).values,
            target_list[2]["power" + var_append].sel(time=slice("1941-03", None)).sel(region=region).values,
            target_list[3]["shortfall" + var_append].sel(time=slice("1941-03", None)).sel(region=region).values,
            predictor_list[0]["nino34" + var_append].sel(time=slice("1941-03", None)).values,
            predictor_list[1]["dmi" + var_append].sel(time=slice("1941-03", None)).values,
            predictor_list[2]["sam" + var_append].sel(time=slice("1941-03", None)).values,
            predictor_list[3]["t2m" + var_append].sel(time=slice("1941-03", None)).sel(region=region).values
        ]
    
    df = pd.DataFrame(
        np.array(data).T,
        columns=["Year", "Month", "Demand", "Wind", "Solar", "Residual_load", "Nino34", "DMI", "SAM", "t2m"]
    )
    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)

    # To CSV
    fp = "/g/data/w42/dr6273/work/projects/Aus_energy/for_SH/monthly_data/"
    if anomalise_monthly:
        filename = scenario + "_monthly_energy_climate_modes_" + region + var_append + "_anomalies.csv"
    else:
        filename = scenario + "_monthly_energy_climate_modes_" + region + var_append + ".csv"
        
    df.to_csv(
        fp + filename, sep=",",
        header=True,
        index=False,
        mode="w"
    )
    
    return df

In [83]:
to_process = {
    "current": [
        demand_current,
        wind_current,
        solar_current,
        shortfall_current
    ],
    "SC2050": [
        demand_SC50,
        wind_SC50,
        solar_SC50,
        shortfall_SC50
    ]
}

In [85]:
for region in wind_current.region.values:
    for anomalise in [True]:
        for scenario, ds_list in zip(to_process.keys(), to_process.values()):

            _df = to_df(
                True,
                region,
                ds_list,
                [
                    nino34,
                    dmi,
                    sam,
                    t2m
                ],
                scenario,
                anomalise_monthly=anomalise
            )